In [1]:
from pathlib import Path
import pandas as pd
import trimesh

In [2]:
SUMMARY_CSV = Path(
    "/home/bsc18/workplace/Jongwon/liver_inr_reconstruction/outputs/meta_vs_random_comparison/siren_meta_vs_random_single_run/summary.csv"
)

OUTPUT_ROOT = SUMMARY_CSV.parent

CASE_NAME = "s0004"

META_EPOCHS = [0, 1, 2, 5, 10, 20, 50, 100, 300]

In [6]:
def load_summary(summary_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(summary_csv)
    df = df[df["status"] == "ok"].copy()
    df["eval_epoch"] = pd.to_numeric(df["eval_epoch"], errors="coerce").astype(int)
    return df


def load_mesh(mesh_path: Path) -> trimesh.Trimesh:
    mesh = trimesh.load_mesh(mesh_path, process=False)

    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(
            [geom for geom in mesh.geometry.values()]
        )

    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Could not load mesh as trimesh.Trimesh: {mesh_path}")

    return mesh


def get_pred_mesh_path(
    output_root: Path,
    init_type: str,
    case_name: str,
    epoch: int,
) -> Path:
    mesh_path = (
        output_root
        / init_type
        / case_name
        / f"epoch_{epoch:04d}"
        / "pred_mesh.stl"
    )

    if not mesh_path.exists():
        raise FileNotFoundError(f"Predicted mesh not found: {mesh_path}")

    return mesh_path


def get_gt_mesh_path(df_case: pd.DataFrame) -> Path:
    gt_paths = df_case["gt_mesh_path"].dropna().unique()

    if len(gt_paths) == 0:
        raise ValueError("No gt_mesh_path found for this case.")

    gt_path = Path(gt_paths[0])

    if not gt_path.exists():
        raise FileNotFoundError(f"Ground-truth mesh not found: {gt_path}")

    return gt_path


def show_mesh(mesh_path: Path):
    mesh = load_mesh(mesh_path)

    print(mesh)
    print("Watertight:", mesh.is_watertight)
    print("Is volume:", mesh.is_volume)
    print("Bounds:\n", mesh.bounds)

    mesh.show()

    return mesh

In [4]:
df = load_summary(SUMMARY_CSV)

df_case = df[df["case_name"] == CASE_NAME].copy()

if len(df_case) == 0:
    raise ValueError(f"No rows found for case: {CASE_NAME}")

available_meta_epochs = sorted(
    df_case[df_case["init_type"] == "meta"]["eval_epoch"].unique()
)

random_rows = df_case[df_case["init_type"] == "random"].copy()

if len(random_rows) == 0:
    raise ValueError(f"No random rows found for case: {CASE_NAME}")

random_final_epoch = int(random_rows["eval_epoch"].max())

print("Case:", CASE_NAME)
print("Available meta epochs:", available_meta_epochs)
print("Random final epoch:", random_final_epoch)

Case: s0004
Available meta epochs: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(70), np.int64(100), np.int64(150), np.int64(200), np.int64(300)]
Random final epoch: 600


In [8]:
gt_mesh_path = get_gt_mesh_path(df_case)

print("Ground truth mesh:")
print(gt_mesh_path)

gt_mesh = show_mesh(gt_mesh_path)

Ground truth mesh:
/home/bsc18/workplace/dataset/Totalsegmentator_dataset_v201_Liver/ok/sampled_20000/mesh/s0004_Segment_1.stl
<trimesh.Trimesh(vertices.shape=(388908, 3), faces.shape=(129636, 3))>
Watertight: False
Is volume: False
Bounds:
 [[-0.47499999 -0.31619146 -0.32873264]
 [ 0.27397817  0.30932206  0.24716544]]


In [9]:
random_mesh_path = get_pred_mesh_path(
    OUTPUT_ROOT,
    "random",
    CASE_NAME,
    random_final_epoch,
)

print("Random INR final mesh:")
print(random_mesh_path)

random_mesh = show_mesh(random_mesh_path)

Random INR final mesh:
/home/bsc18/workplace/Jongwon/liver_inr_reconstruction/outputs/meta_vs_random_comparison/siren_meta_vs_random_single_run/random/s0004/epoch_0600/pred_mesh.stl
<trimesh.Trimesh(vertices.shape=(922560, 3), faces.shape=(307520, 3))>
Watertight: False
Is volume: False
Bounds:
 [[-0.50196075 -0.50196075 -0.50196075]
 [ 0.50192392  0.50192094  0.50192094]]


In [10]:
SELECTED_META_EPOCH = 10

meta_mesh_path = get_pred_mesh_path(
    OUTPUT_ROOT,
    "meta",
    CASE_NAME,
    SELECTED_META_EPOCH,
)

print(f"Meta-INR mesh at epoch {SELECTED_META_EPOCH}:")
print(meta_mesh_path)

meta_mesh = show_mesh(meta_mesh_path)

Meta-INR mesh at epoch 10:
/home/bsc18/workplace/Jongwon/liver_inr_reconstruction/outputs/meta_vs_random_comparison/siren_meta_vs_random_single_run/meta/s0004/epoch_0010/pred_mesh.stl
<trimesh.Trimesh(vertices.shape=(663600, 3), faces.shape=(221200, 3))>
Watertight: False
Is volume: False
Bounds:
 [[-0.49496126 -0.31708321 -0.50002474]
 [ 0.39507303  0.3071264   0.24960944]]


In [11]:
def show_meta_epoch(epoch: int):
    if epoch not in available_meta_epochs:
        raise ValueError(
            f"Meta epoch {epoch} is not available. "
            f"Available epochs: {available_meta_epochs}"
        )

    mesh_path = get_pred_mesh_path(
        OUTPUT_ROOT,
        "meta",
        CASE_NAME,
        epoch,
    )

    print(f"Meta-INR mesh | case={CASE_NAME} | epoch={epoch}")
    print(mesh_path)

    return show_mesh(mesh_path)

In [12]:
show_meta_epoch(10)

Meta-INR mesh | case=s0004 | epoch=10
/home/bsc18/workplace/Jongwon/liver_inr_reconstruction/outputs/meta_vs_random_comparison/siren_meta_vs_random_single_run/meta/s0004/epoch_0010/pred_mesh.stl
<trimesh.Trimesh(vertices.shape=(663600, 3), faces.shape=(221200, 3))>
Watertight: False
Is volume: False
Bounds:
 [[-0.49496126 -0.31708321 -0.50002474]
 [ 0.39507303  0.3071264   0.24960944]]


<trimesh.Trimesh(vertices.shape=(663600, 3), faces.shape=(221200, 3))>

In [13]:
import ipywidgets as widgets
from IPython.display import display

In [14]:
epoch_dropdown = widgets.Dropdown(
    options=available_meta_epochs,
    value=available_meta_epochs[0],
    description="Meta epoch:",
)

show_button = widgets.Button(
    description="Show mesh",
    button_style="primary",
)

output = widgets.Output()


def on_show_clicked(_):
    with output:
        output.clear_output()
        show_meta_epoch(epoch_dropdown.value)


show_button.on_click(on_show_clicked)

display(widgets.HBox([epoch_dropdown, show_button]))
display(output)

Output()

In [15]:
def show_gt_vs_meta(epoch: int):
    if epoch not in available_meta_epochs:
        raise ValueError(
            f"Meta epoch {epoch} is not available. "
            f"Available epochs: {available_meta_epochs}"
        )

    gt = load_mesh(gt_mesh_path)

    meta_path = get_pred_mesh_path(
        OUTPUT_ROOT,
        "meta",
        CASE_NAME,
        epoch,
    )

    meta = load_mesh(meta_path)

    gt.visual.face_colors = [180, 180, 180, 90]      # translucent gray
    meta.visual.face_colors = [230, 80, 60, 210]    # red/orange

    scene = trimesh.Scene()
    scene.add_geometry(gt, node_name="Ground Truth")
    scene.add_geometry(meta, node_name=f"Meta Epoch {epoch}")

    print(f"Ground truth vs Meta-INR epoch {epoch}")
    print("Gray = ground truth")
    print("Red = meta reconstruction")

    scene.show()

    return scene

In [16]:
show_gt_vs_meta(10)

Ground truth vs Meta-INR epoch 10
Gray = ground truth
Red = meta reconstruction


<trimesh.Scene(len(geometry)=2)>

In [17]:
def show_random_vs_meta(epoch: int):
    if epoch not in available_meta_epochs:
        raise ValueError(
            f"Meta epoch {epoch} is not available. "
            f"Available epochs: {available_meta_epochs}"
        )

    random_path = get_pred_mesh_path(
        OUTPUT_ROOT,
        "random",
        CASE_NAME,
        random_final_epoch,
    )

    meta_path = get_pred_mesh_path(
        OUTPUT_ROOT,
        "meta",
        CASE_NAME,
        epoch,
    )

    random_mesh = load_mesh(random_path)
    meta_mesh = load_mesh(meta_path)

    random_mesh.visual.face_colors = [60, 120, 220, 130]   # blue
    meta_mesh.visual.face_colors = [230, 80, 60, 210]      # red/orange

    scene = trimesh.Scene()
    scene.add_geometry(random_mesh, node_name=f"Random Epoch {random_final_epoch}")
    scene.add_geometry(meta_mesh, node_name=f"Meta Epoch {epoch}")

    print(f"Random INR epoch {random_final_epoch} vs Meta-INR epoch {epoch}")
    print("Blue = random final")
    print("Red = meta checkpoint")

    scene.show()

    return scene

In [18]:
import numpy as np

In [19]:
def translated_copy(mesh: trimesh.Trimesh, offset):
    copied = mesh.copy()
    copied.apply_translation(offset)
    return copied


def show_progression_scene(meta_epochs=None):
    if meta_epochs is None:
        meta_epochs = [0, 1, 2, 5, 10, 20, 50, 100, 300]

    scene = trimesh.Scene()

    spacing = 1.4
    index = 0

    # Ground truth
    gt = load_mesh(gt_mesh_path)
    gt.visual.face_colors = [180, 180, 180, 255]
    gt_shifted = translated_copy(gt, [index * spacing, 0, 0])
    scene.add_geometry(gt_shifted, node_name="Ground Truth")
    index += 1

    # Random final
    random_path = get_pred_mesh_path(
        OUTPUT_ROOT,
        "random",
        CASE_NAME,
        random_final_epoch,
    )

    random_mesh = load_mesh(random_path)
    random_mesh.visual.face_colors = [60, 120, 220, 255]
    random_shifted = translated_copy(random_mesh, [index * spacing, 0, 0])
    scene.add_geometry(random_shifted, node_name=f"Random Epoch {random_final_epoch}")
    index += 1

    # Meta checkpoints
    for epoch in meta_epochs:
        if epoch not in available_meta_epochs:
            print(f"Skipping unavailable meta epoch: {epoch}")
            continue

        meta_path = get_pred_mesh_path(
            OUTPUT_ROOT,
            "meta",
            CASE_NAME,
            epoch,
        )

        meta_mesh = load_mesh(meta_path)
        meta_mesh.visual.face_colors = [230, 80, 60, 255]
        meta_shifted = translated_copy(meta_mesh, [index * spacing, 0, 0])

        scene.add_geometry(meta_shifted, node_name=f"Meta Epoch {epoch}")
        index += 1

    print("Scene layout:")
    print("Gray = ground truth")
    print("Blue = random final")
    print("Red = meta checkpoints")
    print("Meshes are translated along the x-axis for side-by-side comparison.")

    scene.show()

    return scene

In [20]:
show_progression_scene([0, 1, 2, 5, 10, 20, 50, 100, 300])

Scene layout:
Gray = ground truth
Blue = random final
Red = meta checkpoints
Meshes are translated along the x-axis for side-by-side comparison.


<trimesh.Scene(len(geometry)=11)>